In [1]:
# --- headless execution harness (added by run_experiments.py) ---
import os as _h_os
import matplotlib as _h_mpl
_h_mpl.use('Agg')
import matplotlib.pyplot as _h_plt
_OUT = 'C:\\Users\\ASUS\\group_sgu26_deep_learning\\BTL03\\output\\4. Training Utilities\\3_Emotion_Reg_CNN_pytorch'
_h_os.makedirs(_OUT, exist_ok=True)
_h_n = [0]
_h_show = _h_plt.show
def _saving_show(*args, **kwargs):
    _figs = [_h_plt.figure(n) for n in _h_plt.get_fignums()]
    for _f in _figs:
        _h_n[0] += 1
        _f.savefig(_h_os.path.join(_OUT, 'fig_%02d.png' % _h_n[0]), dpi=110, bbox_inches='tight')
    _h_show(*args, **kwargs)
    for _f in _figs:
        _h_plt.close(_f)
_h_plt.show = _saving_show
print('[harness] figures -> ' + _OUT)


[harness] figures -> C:\Users\ASUS\group_sgu26_deep_learning\BTL03\output\4. Training Utilities\3_Emotion_Reg_CNN_pytorch


In [2]:
# Demo (PyTorch)

# Overview
### Data augmentation + EarlyStopping to avoid overfitting (PyTorch)
- FER2013 dataset (face images + facial expression)
  + Goal: classifying facial expression
  + Input: a 48x48 image (PyTorch format: `(1, 48, 48)`)
  + Output: facial expression (`angry`, `disgust`, `fear`, `happy`, `sad`, `surprise`, `neutral`)
- Data augmentation via `torchvision.transforms` (`RandomAffine` + `RandomHorizontalFlip`) tương đương `ImageDataGenerator`
- EarlyStopping (`patience=5`) theo `val_loss`

# Import libraries

In [3]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [4]:
num_classes = 7   # angry, disgust, fear, happy, sad, surprise, neutral

# (1) Data

### Download data at https://drive.google.com/drive/folders/0BxINLo5jshCRbXpRcnl2SjJkdlk?resourcekey=0-7qr7Ro7hgkmFknsN1yParQ&usp=sharing
This source code and `.csv` files MUST be in the same location

In [5]:
trainingset = np.loadtxt('fer2013_training_onehot.csv', delimiter=',')
testingset = np.loadtxt('fer2013_publictest_onehot.csv', delimiter=',')

print('trainingset shape:', trainingset.shape)
print('testingset shape :', testingset.shape)

trainingset shape: (28709, 2311)
testingset shape : (3589, 2311)


In [6]:
n_inputs = 2304
n_classes = 7
img_dim = 48

x_training = trainingset[:, 0:n_inputs].astype(np.float32)
y_training_oh = trainingset[:, n_inputs:n_inputs + n_classes]

x_testing = testingset[:, 0:n_inputs].astype(np.float32)
y_testing_oh = testingset[:, n_inputs:n_inputs + n_classes]

# Định dạng NCHW (N, 1, 48, 48)
x_training = x_training.reshape(-1, 1, img_dim, img_dim)
x_testing = x_testing.reshape(-1, 1, img_dim, img_dim)

y_training = np.argmax(y_training_oh, axis=1).astype(np.int64)
y_testing = np.argmax(y_testing_oh, axis=1).astype(np.int64)

print('x_training:', x_training.shape, '| y_training:', y_training.shape)
print('x_testing :', x_testing.shape,  '| y_testing :', y_testing.shape)

x_training: (28709, 1, 48, 48) | y_training: (28709,)
x_testing : (3589, 1, 48, 48) | y_testing : (3589,)


# (2) Declare model

In [7]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.conv1_1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)    # same  -> (64, 48, 48)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=0)   # valid -> (64, 46, 46)
        self.pool1 = nn.MaxPool2d(2, 2)                              #       -> (64, 23, 23)
        self.conv_dropout1 = nn.Dropout(0.25)

        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)  # same  -> (128, 23, 23)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=0) # valid -> (128, 21, 21)
        self.conv2_3 = nn.Conv2d(128, 218, kernel_size=3, padding=1) # same  -> (218, 21, 21)
        self.conv2_4 = nn.Conv2d(218, 218, kernel_size=3, padding=0) # valid -> (218, 19, 19)
        self.pool2 = nn.MaxPool2d(2, 2)                              #       -> (218, 9, 9)
        self.conv_dropout2 = nn.Dropout(0.25)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(218 * 9 * 9, 1024)
        self.fc_dropout1 = nn.Dropout(0.5)
        self.fc3_7ways = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.conv_dropout1(self.pool1(x))

        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = F.relu(self.conv2_3(x))
        x = F.relu(self.conv2_4(x))
        x = self.conv_dropout2(self.pool2(x))

        x = self.flatten(x)
        x = self.fc_dropout1(F.relu(self.fc1(x)))
        return self.fc3_7ways(x)

model = EmotionCNN(num_classes=num_classes).to(device)
print(model)
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')  # Khớp chính xác 19,028,287 params

EmotionCNN(
  (conv1_1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv1_2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_dropout1): Dropout(p=0.25, inplace=False)
  (conv2_1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2_2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1))
  (conv2_3): Conv2d(128, 218, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2_4): Conv2d(218, 218, kernel_size=(3, 3), stride=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_dropout2): Dropout(p=0.25, inplace=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=17658, out_features=1024, bias=True)
  (fc_dropout1): Dropout(p=0.5, inplace=False)
  (fc3_7ways): Linear(in_features=1024, out_features=7, bias=True)
)
Total params: 19,028,287


In [8]:
try:
    from torchinfo import summary
    print(summary(model, input_size=(1, 1, 48, 48), device=device))
except ImportError:
    print('Cài đặt torchinfo nếu muốn xem bảng summary chi tiết: pip install torchinfo')

Layer (type:depth-idx)                   Output Shape              Param #
EmotionCNN                               [1, 7]                    --
├─Conv2d: 1-1                            [1, 64, 48, 48]           640
├─Conv2d: 1-2                            [1, 64, 46, 46]           36,928
├─MaxPool2d: 1-3                         [1, 64, 23, 23]           --
├─Dropout: 1-4                           [1, 64, 23, 23]           --
├─Conv2d: 1-5                            [1, 128, 23, 23]          73,856
├─Conv2d: 1-6                            [1, 128, 21, 21]          147,584
├─Conv2d: 1-7                            [1, 218, 21, 21]          251,354
├─Conv2d: 1-8                            [1, 218, 19, 19]          427,934
├─MaxPool2d: 1-9                         [1, 218, 9, 9]            --
├─Dropout: 1-10                          [1, 218, 9, 9]            --
├─Flatten: 1-11                          [1, 17658]                --
├─Linear: 1-12                           [1, 1024]           

# (3) Train defined model (Data Augmentation + EarlyStopping)

In [9]:
# Data augmentation tương đương Keras ImageDataGenerator(
#     rotation_range=45, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True
# )
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=45, translate=(0.2, 0.2)),
])

class FER2013Dataset(Dataset):
    def __init__(self, x_array, y_array, transform=None):
        self.x = torch.from_numpy(x_array)
        self.y = torch.from_numpy(y_array)
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        if self.transform is not None:
            img = self.transform(img)
        return img, self.y[idx]

In [10]:
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)
            total_loss += loss.item() * bx.size(0)
            correct += (logits.argmax(dim=1) == by).sum().item()
            total += bx.size(0)
    return total_loss / total, correct / total


lr = 0.0001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

early_stopping_after_epochs = 5
best_val_loss = float('inf')
no_improve_epochs = 0

batch_size = 128
epochs = 100

train_loader = DataLoader(
    FER2013Dataset(x_training, y_training, transform=train_transform),
    batch_size=batch_size, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_testing), torch.from_numpy(y_testing)),
    batch_size=batch_size, shuffle=False
)

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

starting_time = time.time()
for epoch in range(1, epochs + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * bx.size(0)
        correct += (logits.argmax(dim=1) == by).sum().item()
        total += bx.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)

    print(f'Epoch {epoch:3d}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve_epochs = 0
        torch.save(model.state_dict(), 'best_emotion_aug.pt')
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= early_stopping_after_epochs:
            print(f'EarlyStopping triggered at epoch {epoch}.')
            break

print('> training time is %.4f minutes' % ((time.time() - starting_time) / 60))

Epoch   1/100 - loss: 1.8388 - acc: 0.2472 - val_loss: 1.8176 - val_acc: 0.2497


Epoch   2/100 - loss: 1.8095 - acc: 0.2516 - val_loss: 1.7957 - val_acc: 0.2552


Epoch   3/100 - loss: 1.7920 - acc: 0.2589 - val_loss: 1.7509 - val_acc: 0.3023


Epoch   4/100 - loss: 1.7618 - acc: 0.2728 - val_loss: 1.6932 - val_acc: 0.3207


Epoch   5/100 - loss: 1.7226 - acc: 0.2999 - val_loss: 1.6048 - val_acc: 0.3525


Epoch   6/100 - loss: 1.6760 - acc: 0.3269 - val_loss: 1.4791 - val_acc: 0.4271


... (46 outputs left out)

Epoch  53/100 - loss: 1.0098 - acc: 0.6205 - val_loss: 1.0116 - val_acc: 0.6241


Epoch  54/100 - loss: 1.0053 - acc: 0.6228 - val_loss: 0.9914 - val_acc: 0.6344


Epoch  55/100 - loss: 1.0032 - acc: 0.6250 - val_loss: 1.0093 - val_acc: 0.6325


Epoch  56/100 - loss: 1.0086 - acc: 0.6202 - val_loss: 1.0083 - val_acc: 0.6300
EarlyStopping triggered at epoch 56.
> training time is 43.2702 minutes


# (4) Evaluate trained model

In [11]:
# Tùy chọn: nạp lại trọng số tốt nhất đã lưu trong quá trình huấn luyện
# model.load_state_dict(torch.load('best_emotion_aug.pt', map_location=device))

test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print('Test loss:', test_loss)
print('Test accuracy:', test_acc)

Test loss: 1.0083182933736425
Test accuracy: 0.6299804959598774


In [12]:
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

def get_emotion(ohv_or_idx):
    if np.ndim(ohv_or_idx) == 0:
        indx = int(ohv_or_idx)
    else:
        indx = int(np.argmax(ohv_or_idx))
    return EMOTIONS[indx]

In [13]:
img_indx = int(np.random.randint(0, x_testing.shape[0]))
sample = x_testing[img_indx]  # (1, 48, 48)

model.eval()
with torch.no_grad():
    inp = torch.from_numpy(np.expand_dims(sample, axis=0)).to(device)
    pred_probs = F.softmax(model(inp), dim=1).cpu().numpy()

plt.imshow(sample[0], cmap='gray')
plt.show()
print('> testing image index: %d\n> true emotion: %s\n> predicted emotion: %s' % (
    img_indx, get_emotion(y_testing[img_indx]), get_emotion(pred_probs)
))

> testing image index: 3403
> true emotion: angry
> predicted emotion: angry


C:\Users\ASUS\AppData\Local\Temp\ipykernel_15844\4240293948.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  _h_show(*args, **kwargs)
